# 3D · Full EDA — Returns, Drawdowns, Rolling Windows
### Financial Analytics — Module 3

The capstone of Module 3: a complete exploratory analysis of an index, ending with the four charts every market analyst produces on day one of looking at any asset:

1. Price with moving averages
2. Return distribution (meet the fat tails)
3. Drawdown ("underwater") curve
4. Rolling volatility (watch the regime change)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"])

# Set the date as the index - time-series operations expect this
px = px.set_index("date").sort_index()
px.head(3)

## Clean-up first (3B habits, 60 seconds)

The registry lists this file's defects. Handle the one that would poison our stats — the fat-finger `high` — and note the rest.

In [ ]:
# The 2024-08-13 high is 10x reality. We only use 'close' below, but fix it anyway - 
# leaving known-bad values in a dataset you're sharing is how the next person gets hurt.
bad = px.loc["2024-08-13", "high"]
px.loc["2024-08-13", "high"] = px.loc["2024-08-13", "close"] * 1.005
print(f"Fixed fat-finger high: {bad:,.0f} -> {px.loc['2024-08-13','high']:,.2f}")

# The two missing trading days and 2021's missing volume: NOTED, not fixed - 
# they don't affect close-based analysis. Every decision documented. That's the discipline.

---
## 1. Daily returns — the atom of finance

Prices tell you where you are; **returns** tell you what happened. Almost everything downstream (volatility, correlation, Sharpe, VaR) is computed on returns, not prices.

In [ ]:
px["ret"] = px["close"].pct_change()

ann_ret = px["ret"].mean() * 252            # ~252 trading days per year
ann_vol = px["ret"].std() * np.sqrt(252)    # volatility scales with SQRT of time

print(f"Days                : {px['ret'].count()}")
print(f"Best day            : {px['ret'].max()*100:+.2f}%")
print(f"Worst day           : {px['ret'].min()*100:+.2f}%")
print(f"Annualised return   : {ann_ret*100:.1f}%")
print(f"Annualised volatility: {ann_vol*100:.1f}%")

Two conventions to memorise: **×252** to annualise a mean daily return, **×√252** to annualise daily volatility. The square root is because independent risks add in *variance*, not in standard deviation — Module 10 will make you feel this via simulation.

---
## 2. Price + moving averages

In [ ]:
px["ma50"] = px["close"].rolling(50).mean()      # rolling = a sliding window
px["ma200"] = px["close"].rolling(200).mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(px.index, px["close"], lw=1, label="NIFTY 50", color="#2563EB")
ax.plot(px.index, px["ma50"], lw=1.2, label="50-day MA", color="#EA580C")
ax.plot(px.index, px["ma200"], lw=1.2, label="200-day MA", color="#0D9488")
ax.set_title("NIFTY 50 with moving averages")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

`rolling(50).mean()` slides a 50-day window along the series — the first 49 values are NaN because the window isn't full yet. Crossovers of these two averages are the basis of the strategy you'll backtest in Lab 3 (Module 11).

---
## 3. The return distribution — meet the fat tails

In [ ]:
from scipy import stats as st

r = px["ret"].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(r, bins=80, density=True, alpha=0.65, color="#2563EB", label="Actual daily returns")

# Overlay the normal curve with the SAME mean and std
x = np.linspace(r.min(), r.max(), 300)
ax.plot(x, st.norm.pdf(x, r.mean(), r.std()), "r-", lw=2, label="Normal (same mean & std)")
ax.set_title("Daily returns vs the bell curve")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# How often "should" a 3-sigma day happen vs how often it DID?
sigma3 = (r.abs() > 3 * r.std()).sum()
expected = len(r) * 2 * (1 - st.norm.cdf(3))
print(f"3-sigma days - expected under normality: {expected:.1f} | actual: {sigma3}")
print(f"Kurtosis (normal = 0): {st.kurtosis(r):.2f}  -> fat tails confirmed")

**This is the most important chart in quantitative finance.** Real markets produce extreme days far more often than the bell curve allows. Models that assume normality understate risk *precisely when it matters*. Keep this chart in mind through Module 10 (simulation) and Module 12 (portfolio risk).

---
## 4. Drawdown — the pain chart

A drawdown is how far you are below the highest point so far. It's the chart that answers the investor's real question: *how bad did it get, and for how long?*

In [ ]:
cummax = px["close"].cummax()                 # highest close so far
px["drawdown"] = px["close"] / cummax - 1     # % below that peak

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.fill_between(px.index, px["drawdown"] * 100, 0, color="#DC2626", alpha=0.5)
ax.set_title("Drawdown from peak (%)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

trough = px["drawdown"].idxmin()
print(f"Max drawdown: {px['drawdown'].min()*100:.1f}% (trough on {trough.date()})")

Three lines: `cummax`, a division, a fill chart. This exact exhibit appears in every fund factsheet on earth.

---
## 5. Rolling volatility — watch the regime change

In [ ]:
px["vol_60d"] = px["ret"].rolling(60).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(px.index, px["vol_60d"], color="#7C3AED", lw=1.2)
ax.set_title("60-day rolling volatility, annualised (%)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**See the step up partway through?** The registry told you this series has a volatility regime change built in. A single 'average volatility' number for this whole period describes *neither* regime — Module 1's fourth bias, now visible as a picture instead of a warning.

---
## 6. Real data — prove it on the live market

Everything above ran on course data. Same code, real NIFTY, via yfinance:

In [ ]:
# Requires internet. In Colab this just works after the pip install.
# !pip install yfinance --quiet
try:
    import yfinance as yf
    real = yf.download("^NSEI", start="2023-01-01", progress=False)
    real_ret = real["Close"].pct_change().dropna()
    print(f"Live NIFTY days fetched : {len(real)}")
    print(f"Annualised vol (real)   : {float(real_ret.std().iloc[0] if hasattr(real_ret.std(), 'iloc') else real_ret.std()) * np.sqrt(252) * 100:.1f}%")
except Exception as e:
    print("Offline or yfinance unavailable - fine, the course data covers everything above.")
    print(f"({type(e).__name__})")

### ✏️ Exercises
1. **Best/worst month:** with the date index, `px["ret"].resample("ME").sum()` gives monthly returns. Which month was worst, and does it line up with the drawdown trough?
2. **Volume check:** compute average daily volume per year. What happens for 2021, and what does `.mean()` silently do about it? (Check the docs for `skipna` — silent behaviour you should know about.)
3. **Drawdown duration:** how many total days did the index spend more than 10% below its peak? (Hint: a boolean mask and `.sum()`.)
4. **Real-data rerun:** if you have internet, rerun sections 2–5 on the yfinance data. Which parts of the code needed zero changes? (Answer: that's the point of writing analysis as code.)

---
**Module 3 complete.** You can now load, inspect, clean, join, aggregate and explore financial data — the core competency of every analyst job. Claim your **Data Wrangler** badge.

*AI disclosure: ______*